In [0]:
# Dependency cell (added during notebook split).
# Imports used by the cells below — mirrors the wildcard import in the original
# olist_full_backup notebook (cell 14) so this notebook runs standalone.
from pyspark.sql.functions import *

In [0]:
# Dependency cell (added during notebook split).
# Load the Silver tables produced by 02_silver_transformations and the Gold tables
# produced by 03_gold_analytics. The temporary SQL views used by the sections below
# (incoming_orders, updated_orders, fixed_updated_orders, silver_orders, golden_order_summary,
# silver_order_items, silver_sellers, silver_customers, gold_order_analytics) are (re)created
# within their own cells when this notebook is run in order.
silver_orders = spark.table('workspace.olist.silver_orders')
silver_order_items = spark.table('workspace.olist.silver_order_items')
silver_customers = spark.table('workspace.olist.silver_customers')
silver_sellers = spark.table('workspace.olist.silver_sellers')
golden_order_summary = spark.table('workspace.olist.golden_order_summary')
gold_order_analytics = spark.table('workspace.olist.gold_order_analytics')
gold_seller_analytics = spark.table('workspace.olist.gold_seller_analytics')
gold_customer_analytics = spark.table('workspace.olist.gold_customer_analytics')

# End to End validation

## Validation Overview

This section performs end-to-end validation of the Olist pipeline: the raw data landed in the Bronze layer, the cleaned and standardized tables in Silver, and the analytical aggregates in Gold must all describe the same underlying business reality. A medallion pipeline is only trustworthy if each layer can be reconciled against the ones before it, so this section runs a series of SQL checks that trace the data all the way from landing to analytics.

The validation proceeds in four stages:

1. **Bronze-to-Silver reconciliation** — row counts of each `bronze_*` table are compared with their `silver_*` counterpart to confirm no records were lost or duplicated during transformation.
2. **Primary key and uniqueness checks** — the primary key of each Silver table is tested for nulls and duplicates, because every downstream join and aggregation assumes those keys are unique and complete.
3. **Join integrity checks** — the relationships between `silver_orders`, `silver_order_items`, `silver_products`, `silver_customers` and `silver_sellers` are validated with `LEFT JOIN` counts of unmatched records.
4. **Gold metrics reconciliation** — headline metrics stored in `gold_order_analytics`, `gold_product_analytics`, `gold_seller_analytics` and `gold_customer_analytics` are recomputed from `silver_order_items` and compared with the Gold values.

The section opens with `SHOW TABLES IN workspace.olist` to confirm which tables exist before the checks run against them.

In [0]:
%sql
SHOW TABLES IN workspace.olist;

## Bronze-to-Silver Data Reconciliation

This check compares row counts between each Bronze table and its Silver counterpart for all five datasets — orders, order_items, products, customers and sellers.

Row counts are the simplest and cheapest reconciliation check: they verify that the transformation stage preserved record cardinality. The Silver transformations standardize and clean the Bronze data, so each Bronze record is expected to appear exactly once in Silver. A mismatch points to a problem in either direction:

* **Silver count lower than Bronze** — records were dropped or filtered during transformation.
* **Silver count higher than Bronze** — records were duplicated, e.g. by a join or union applied more than once.

Note that the Bronze orders table lives in `workspace.default.bronze_orders` while all other Bronze and Silver tables live in the `workspace.olist` schema — the query references each location explicitly.

Matching row counts between the `bronze_*` and `silver_*` tables indicate that no data was lost and nothing was duplicated in the move from raw ingestion to the cleaned layer. They do not by themselves prove column-level correctness, which is why the key and join checks that follow are also needed.

In [0]:
%sql
SELECT
    'orders' AS dataset,
    (SELECT COUNT(*) FROM workspace.default.bronze_orders) AS bronze_count,
    (SELECT COUNT(*) FROM workspace.olist.silver_orders) AS silver_count

UNION ALL

SELECT
    'order_items',
    (SELECT COUNT(*) FROM workspace.olist.bronze_order_items),
    (SELECT COUNT(*) FROM workspace.olist.silver_order_items)

UNION ALL

SELECT
    'products',
    (SELECT COUNT(*) FROM workspace.olist.bronze_products),
    (SELECT COUNT(*) FROM workspace.olist.silver_products)

UNION ALL

SELECT
    'customers',
    (SELECT COUNT(*) FROM workspace.olist.bronze_customers),
    (SELECT COUNT(*) FROM workspace.olist.silver_customers)

UNION ALL

SELECT
    'sellers',
    (SELECT COUNT(*) FROM workspace.olist.bronze_sellers),
    (SELECT COUNT(*) FROM workspace.olist.silver_sellers);

## Primary Key and Uniqueness Validation

Every join and aggregation in the Gold layer assumes the Silver tables have reliable primary keys. This check counts total rows, distinct keys and null keys for each of the five Silver datasets:

* `silver_orders` — primary key: `order_id`
* `silver_order_items` — composite key: `order_id` together with `order_item_id`, tested via `CONCAT(order_id, '_', order_item_id)`
* `silver_products` — primary key: `product_id`
* `silver_customers` — primary key: `customer_id`
* `silver_sellers` — primary key: `seller_id`

Null and duplicate keys must be ruled out because they break downstream logic in different ways: a null key can never be joined to its related tables, silently orphaning that row in any Gold aggregation, while duplicate keys multiply rows in joins — an order joined against a duplicated `order_id` inflates revenue and item counts — and make key-based `MERGE` operations ambiguous. `order_items` is the one table with a composite key: `order_id` legitimately repeats once per item in an order, so uniqueness only holds for the combination of `order_id` and `order_item_id`.

For each dataset, the expected outcome is `unique_keys == total_rows` and `null_keys == 0`.

In [0]:
%sql
SELECT
    'orders' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_keys,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM workspace.olist.silver_orders

UNION ALL

SELECT
    'order_items',
    COUNT(*),
    COUNT(DISTINCT CONCAT(order_id, '_', order_item_id)),
    SUM(CASE WHEN order_id IS NULL OR order_item_id IS NULL THEN 1 ELSE 0 END)
FROM workspace.olist.silver_order_items

UNION ALL

SELECT
    'products',
    COUNT(*),
    COUNT(DISTINCT product_id),
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
FROM workspace.olist.silver_products

UNION ALL

SELECT
    'customers',
    COUNT(*),
    COUNT(DISTINCT customer_id),
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
FROM workspace.olist.silver_customers

UNION ALL

SELECT
    'sellers',
    COUNT(*),
    COUNT(DISTINCT seller_id),
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END)
FROM workspace.olist.silver_sellers;

## Join Integrity Validation

The analytical value of the Olist dataset comes from relating orders to the customers who placed them, the items they contain, and the products and sellers behind those items. This check validates those relationships by `LEFT JOIN`ing the "many" side of each relationship to its parent table and counting the rows that fail to find a match:

* `orders_order_items` — `silver_order_items` joined to `silver_orders` on `order_id`
* `customers_orders` — `silver_orders` joined to `silver_customers` on `customer_id`
* `products_order_items` — `silver_order_items` joined to `silver_products` on `product_id`
* `sellers_order_items` — `silver_order_items` joined to `silver_sellers` on `seller_id`

An `unmatched_records` count greater than zero would indicate a referential integrity break — a foreign key pointing at a parent row that does not exist. Typical causes include an incomplete ingestion of one of the source files, rows dropped during a Silver transformation, or keys altered during cleaning. Unmatched rows here would also distort the Gold layer directly: aggregations that join these tables would silently lose those records, so this check protects every metric built later in the pipeline.

In [0]:
%sql
WITH join_checks AS (

    SELECT
        'orders_order_items' AS join_name,
        COUNT(*) AS joined_rows,
        SUM(CASE WHEN o.order_id IS NULL THEN 1 ELSE 0 END) AS unmatched_records
    FROM workspace.olist.silver_order_items i
    LEFT JOIN workspace.olist.silver_orders o
        ON i.order_id = o.order_id

    UNION ALL

    SELECT
        'customers_orders',
        COUNT(*),
        SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END)
    FROM workspace.olist.silver_orders o
    LEFT JOIN workspace.olist.silver_customers c
        ON o.customer_id = c.customer_id

    UNION ALL

    SELECT
        'products_order_items',
        COUNT(*),
        SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END)
    FROM workspace.olist.silver_order_items i
    LEFT JOIN workspace.olist.silver_products p
        ON i.product_id = p.product_id

    UNION ALL

    SELECT
        'sellers_order_items',
        COUNT(*),
        SUM(CASE WHEN s.seller_id IS NULL THEN 1 ELSE 0 END)
    FROM workspace.olist.silver_order_items i
    LEFT JOIN workspace.olist.silver_sellers s
        ON i.seller_id = s.seller_id
)

SELECT * FROM join_checks;

## Gold Metrics Reconciliation

The Gold layer aggregates the Silver data into analytical tables, and aggregation is where errors compound — a wrong join, a double-counted source, or a mislabeled column can shift totals while every individual row still looks correct. These checks therefore recompute headline metrics directly from `silver_order_items` and compare them with the values already stored in the Gold tables.

The source metrics are computed as: item count (`COUNT(*)`), order count (`COUNT(DISTINCT order_id)`), revenue (`SUM(price)`), freight (`SUM(freight_value)`) and order value (`SUM(item_value)`). They are compared against the stored aggregates:

* `gold_order_analytics` — row count (`total_orders`), `SUM(total_items)`, `SUM(total_price)`, `SUM(total_freight_value)` and `SUM(total_item_value)`
* `gold_product_analytics` — `SUM(total_units_sold)`, `SUM(total_revenue)`, `SUM(total_freight)`, `SUM(total_order_value)`
* `gold_seller_analytics` — `SUM(total_items_sold)`, `SUM(total_revenue)`, `SUM(total_freight)`, `SUM(total_order_value)`
* `gold_customer_analytics` — `SUM(total_orders)`, `SUM(total_items_purchased)`, `SUM(total_spending)`

The comparison query returns a `source_value` and `gold_value` pair for every check — item counts, revenue, freight, order counts and customer spending across all four Gold tables.

Two focused follow-up checks then isolate the most important totals one by one: the `silver_orders` row count against `SUM(total_orders)` from `gold_customer_analytics` (order count), and `SUM(item_value)` from `silver_order_items` against the value totals in `gold_order_analytics`, `gold_product_analytics` and `gold_seller_analytics` (order value).

Revenue, freight, item counts and order counts matter because they are the pipeline's headline business numbers — the figures every dashboard and downstream analysis built on Gold depends on. If Gold disagrees with Silver on any of them, the Gold layer does not represent the source data and the aggregation logic in `03_gold_analytics` needs to be revisited.

In [0]:
%sql

WITH source_metrics AS (
    SELECT
        COUNT(*) AS total_items,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(price), 2) AS revenue,
        ROUND(SUM(freight_value), 2) AS freight,
        ROUND(SUM(item_value), 2) AS order_value
    FROM workspace.olist.silver_order_items
),

order_metrics AS (
    SELECT
        COUNT(*) AS total_orders,
        SUM(total_items) AS total_items,
        ROUND(SUM(total_price), 2) AS revenue,
        ROUND(SUM(total_freight_value), 2) AS freight,
        ROUND(SUM(total_item_value), 2) AS order_value
    FROM workspace.olist.gold_order_analytics
),

product_metrics AS (
    SELECT
        SUM(total_units_sold) AS total_items,
        ROUND(SUM(total_revenue), 2) AS revenue,
        ROUND(SUM(total_freight), 2) AS freight,
        ROUND(SUM(total_order_value), 2) AS order_value
    FROM workspace.olist.gold_product_analytics
),

seller_metrics AS (
    SELECT
        SUM(total_items_sold) AS total_items,
        ROUND(SUM(total_revenue), 2) AS revenue,
        ROUND(SUM(total_freight), 2) AS freight,
        ROUND(SUM(total_order_value), 2) AS order_value
    FROM workspace.olist.gold_seller_analytics
),

customer_metrics AS (
    SELECT
        SUM(total_orders) AS total_orders,
        SUM(total_items_purchased) AS total_items,
        ROUND(SUM(total_spending), 2) AS revenue
    FROM workspace.olist.gold_customer_analytics
)

SELECT 'Order rows' AS validation,
       (SELECT COUNT(*) FROM workspace.olist.silver_orders) AS source_value,
       total_orders AS gold_value
FROM order_metrics

UNION ALL

SELECT 'Order Gold item count',
       total_items,
       (SELECT total_items FROM order_metrics)
FROM source_metrics

UNION ALL

SELECT 'Order Gold revenue',
       revenue,
       (SELECT revenue FROM order_metrics)
FROM source_metrics

UNION ALL

SELECT 'Product Gold item count',
       total_items,
       (SELECT total_items FROM product_metrics)
FROM source_metrics

UNION ALL

SELECT 'Product Gold revenue',
       revenue,
       (SELECT revenue FROM product_metrics)
FROM source_metrics

UNION ALL

SELECT 'Product Gold freight',
       freight,
       (SELECT freight FROM product_metrics)
FROM source_metrics

UNION ALL

SELECT 'Seller Gold item count',
       total_items,
       (SELECT total_items FROM seller_metrics)
FROM source_metrics

UNION ALL

SELECT 'Seller Gold revenue',
       revenue,
       (SELECT revenue FROM seller_metrics)
FROM source_metrics

UNION ALL

SELECT 'Seller Gold freight',
       freight,
       (SELECT freight FROM seller_metrics)
FROM source_metrics

UNION ALL

SELECT 'Customer Gold order count',
       total_orders,
       (SELECT total_orders FROM customer_metrics)
FROM source_metrics

UNION ALL

SELECT 'Customer Gold item count',
       total_items,
       (SELECT total_items FROM customer_metrics)
FROM source_metrics

UNION ALL

SELECT 'Customer Gold spending',
       revenue,
       (SELECT revenue FROM customer_metrics)
FROM source_metrics;

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.olist.silver_orders)
        AS silver_total_orders,

    (SELECT SUM(total_orders)
     FROM workspace.olist.gold_customer_analytics)
        AS gold_customer_total_orders;

In [0]:
%sql
SELECT
    ROUND(SUM(item_value), 2) AS silver_order_value,

    (SELECT ROUND(SUM(total_item_value), 2)
     FROM workspace.olist.gold_order_analytics)
        AS gold_order_value,

    (SELECT ROUND(SUM(total_order_value), 2)
     FROM workspace.olist.gold_product_analytics)
        AS gold_product_value,

    (SELECT ROUND(SUM(total_order_value), 2)
     FROM workspace.olist.gold_seller_analytics)
        AS gold_seller_value

FROM workspace.olist.silver_order_items;

In [0]:
# Read existing Silver orders
orders_df = spark.table('workspace.olist.silver_orders')

# Divide records into two deterministic batches
initial_batch = orders_df.filter(
    pmod(xxhash64(col('order_id')), lit(10)) < 8
)

incoming_batch = orders_df.filter(
    pmod(xxhash64(col('order_id')), lit(10)) >= 8
)

# Save initial batch as the Delta target table
initial_batch.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.incremental_orders_demo')

In [0]:
print("Original orders:", orders_df.count())
print("Initial batch:", initial_batch.count())
print("Incoming batch:", incoming_batch.count())
print(
    "Initial target:",
    spark.table('workspace.olist.incremental_orders_demo').count()
)

In [0]:
# temp view from incoming batch
incoming_batch.createOrReplaceTempView("incoming_orders")

In [0]:
%sql

MERGE INTO workspace.olist.incremental_orders_demo AS target

USING incoming_orders AS source

ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *

In [0]:
# validating final target
merged_orders = spark.table(
    'workspace.olist.incremental_orders_demo'
)

print("Final target count:", merged_orders.count())

print(
    "Distinct order IDs:",
    merged_orders.select('order_id').distinct().count()
)

In [0]:
# Update Batch
target_orders = spark.table(
    'workspace.olist.incremental_orders_demo'
)

# Select 5 existing orders that are not delivered
update_batch = target_orders.filter(
    col('order_status') != 'delivered'
).orderBy('order_id').limit(5)

# Simulate their updated status
update_batch = update_batch.withColumn(
    'order_status',
    lit('delivered')
)

update_batch.createOrReplaceTempView('updated_orders')

In [0]:
update_batch.select(
    'order_id',
    'order_status'
).display()

In [0]:
%sql
-- Merge Updates

MERGE INTO workspace.olist.incremental_orders_demo AS target

USING updated_orders AS source

ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET
        target.order_status = source.order_status

In [0]:
# validating operations
final_orders = spark.table(
    'workspace.olist.incremental_orders_demo'
)

print("Total records:", final_orders.count())

print(
    "Distinct order IDs:",
    final_orders.select('order_id').distinct().count()
)

print(
    "Delivered records:",
    final_orders.filter(
        col('order_status') == 'delivered'
    ).count()
)

In [0]:
# fixed batch
from pyspark.sql.functions import col, lit
target_orders = spark.table(
    'workspace.olist.incremental_orders_demo'
)

fixed_update_batch = target_orders.filter(
    col('order_status') != 'delivered'
).orderBy('order_id').limit(5).withColumn(
    'order_status',
    lit('delivered')
).cache()

fixed_update_batch.count()

fixed_update_batch.createOrReplaceTempView('fixed_updated_orders')

In [0]:
%sql
MERGE INTO workspace.olist.incremental_orders_demo AS target
USING fixed_updated_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED
AND target.order_status <> source.order_status
THEN UPDATE SET
    target.order_status = source.order_status

# Spark Optimization

In [0]:
orders_df = spark.table('workspace.olist.silver_orders')

delivered_orders = orders_df.filter(
    col('order_status') == 'delivered'
).select(
    'order_id',
    'customer_id',
    'order_status'
)

delivered_orders.explain(True)

In [0]:
orders_df = spark.table('workspace.olist.silver_orders')

# Query 1: Read all columns
all_columns = orders_df

# Query 2: Read only required columns
selected_columns = orders_df.select(
    'order_id',
    'customer_id'
)

print("All columns:")
all_columns.explain()

print("Selected columns:")
selected_columns.explain()

In [0]:
from pyspark.sql.functions import spark_partition_id, count

orders_df = spark.table('workspace.olist.silver_orders')

partition_details = orders_df.withColumn(
    'partition_id',
    spark_partition_id()
).groupBy(
    'partition_id'
).agg(
    count('*').alias('record_count')
).orderBy('partition_id')

partition_details.display()

In [0]:
from pyspark.sql.functions import spark_partition_id, count

repartitioned_df = orders_df.repartition(4)

partition_details = repartitioned_df.withColumn(
    'partition_id',
    spark_partition_id()
).groupBy(
    'partition_id'
).agg(
    count('*').alias('record_count')
).orderBy('partition_id')

partition_details.display()

In [0]:
reduced_df = repartitioned_df.coalesce(2)

partition_details = reduced_df.withColumn(
    'partition_id',
    spark_partition_id()
).groupBy(
    'partition_id'
).agg(
    count('*').alias('record_count')
).orderBy('partition_id')

partition_details.display()

In [0]:
orders_df = spark.table('workspace.olist.silver_orders')

# Cache the DataFrame
cached_orders = orders_df.cache()

# First action: materialize the cache
print("Total orders:", cached_orders.count())

In [0]:
from pyspark.sql.functions import col
# Second action: reuse the cached DataFrame
print(
    "Delivered orders:",
    cached_orders.filter(
        col('order_status') == 'delivered'
    ).count()
)

In [0]:
cached_orders.unpersist()

In [0]:
from pyspark.sql.functions import broadcast

order_items = spark.table(
    'workspace.olist.silver_order_items'
)

sellers = spark.table(
    'workspace.olist.silver_sellers'
)

broadcast_join = order_items.join(
    broadcast(sellers),
    on='seller_id',
    how='inner'
)

In [0]:
broadcast_join.explain(True)

In [0]:
print("Joined records:", broadcast_join.count())

print(
    "Distinct sellers:",
    broadcast_join.select('seller_id').distinct().count()
)

In [0]:
from pyspark.sql.functions import sum, round

order_items = spark.table(
    'workspace.olist.silver_order_items'
)

seller_revenue = order_items.groupBy(
    'seller_id'
).agg(
    round(sum('price'), 2).alias('total_revenue')
)

In [0]:
seller_revenue.explain(True)

In [0]:
order_items = spark.table(
    'workspace.olist.silver_order_items'
)

sellers = spark.table(
    'workspace.olist.silver_sellers'
)

normal_join = order_items.join(
    sellers,
    on='seller_id',
    how='inner'
)

normal_join.explain(True)

In [0]:
from pyspark.sql.functions import count, desc

order_items = spark.table('workspace.olist.silver_order_items')

seller_distribution = order_items.groupBy('seller_id').agg(
    count('*').alias('total_items')
)

seller_distribution.orderBy(
    desc('total_items')
).show(10)

In [0]:
from pyspark.sql.functions import avg, max, min, round

seller_distribution.agg(
    round(avg('total_items'), 2).alias('avg_items'),
    max('total_items').alias('max_items'),
    min('total_items').alias('min_items')
).show()

In [0]:
from pyspark.sql.functions import spark_partition_id, count

partition_distribution = order_items.groupBy(
    spark_partition_id().alias('partition_id')
).agg(
    count('*').alias('total_records')
)

partition_distribution.orderBy('partition_id').show()

In [0]:
from pyspark.sql.functions import spark_partition_id, count

partitioned_items = order_items.repartition(4, 'seller_id')

partitioned_items.groupBy(
    spark_partition_id().alias('partition_id')
).agg(
    count('*').alias('total_records')
).orderBy('partition_id').show()

In [0]:
top_seller = seller_distribution.orderBy(
    desc('total_items')
).first()

print(top_seller['seller_id'])
print(top_seller['total_items'])

In [0]:
from pyspark.sql.functions import (
    col, monotonically_increasing_id, pmod, lit
)

target_seller = '6560211a19b47992c3666cc44a7e94c0'

salted_seller = order_items.filter(
    col('seller_id') == target_seller
).withColumn(
    'salt',
    pmod(monotonically_increasing_id(), lit(4))
)

In [0]:
salted_seller.groupBy('salt').count().orderBy('salt').show()


# SQL OPTIMIZATION

In [0]:
spark.table('workspace.olist.silver_orders') \
    .createOrReplaceTempView('silver_orders')

spark.table('workspace.olist.golden_order_summary') \
    .createOrReplaceTempView('golden_order_summary')

In [0]:
gold_order_sql = spark.sql("""
    SELECT
        s.*,
        g.total_items,
        g.total_price,
        g.total_freight_value,
        g.total_item_value
    FROM silver_orders AS s
    LEFT JOIN golden_order_summary AS g
    ON s.order_id = g.order_id
""")

In [0]:
print("SQL row count:", gold_order_sql.count())

print(
    "Distinct orders:",
    gold_order_sql.select('order_id').distinct().count()
)

In [0]:
spark.table('workspace.olist.silver_order_items') \
    .createOrReplaceTempView('silver_order_items')

spark.table('workspace.olist.silver_sellers') \
    .createOrReplaceTempView('silver_sellers')

In [0]:
gold_seller_sql = spark.sql("""
    WITH seller_performance AS (
        SELECT
            seller_id,
            COUNT(*) AS total_items_sold,
            COUNT(DISTINCT order_id) AS total_orders,
            ROUND(SUM(price), 2) AS total_revenue,
            ROUND(SUM(freight_value), 2) AS total_freight,
            ROUND(SUM(price) + SUM(freight_value), 2) AS total_order_value,
            ROUND(AVG(price), 2) AS avg_selling_price
        FROM silver_order_items
        GROUP BY seller_id
    )

    SELECT
        s.seller_id,
        s.seller_zip_code_prefix,
        s.seller_city,
        s.seller_state,
        p.total_items_sold,
        p.total_orders,
        p.total_revenue,
        p.total_freight,
        p.total_order_value,
        p.avg_selling_price
    FROM silver_sellers s
    LEFT JOIN seller_performance p
        ON s.seller_id = p.seller_id
""")

In [0]:
existing_seller = spark.table('workspace.olist.gold_seller_analytics')

print("SQL rows:", gold_seller_sql.count())
print("Existing Gold rows:", existing_seller.count())
print(
    "Distinct sellers:",
    gold_seller_sql.select('seller_id').distinct().count()
)

In [0]:
seller_columns = [
    'seller_id',
    'total_items_sold',
    'total_orders',
    'total_revenue',
    'total_freight',
    'total_order_value',
    'avg_selling_price'
]

sql_difference = gold_seller_sql.select(*seller_columns).exceptAll(
    existing_seller.select(*seller_columns)
)

gold_difference = existing_seller.select(*seller_columns).exceptAll(
    gold_seller_sql.select(*seller_columns)
)

print("SQL records missing from Gold:", sql_difference.count())
print("Gold records missing from SQL:", gold_difference.count())

In [0]:
spark.table('workspace.olist.silver_customers') \
    .createOrReplaceTempView('silver_customers')

spark.table('workspace.olist.gold_order_analytics') \
    .createOrReplaceTempView('gold_order_analytics')

In [0]:
gold_customer_sql = spark.sql("""
    WITH customer_order_details AS (
        SELECT
            c.customer_unique_id,
            o.order_id,
            o.total_item_value,
            o.total_price,
            o.total_items
        FROM silver_customers c
        LEFT JOIN gold_order_analytics o
            ON c.customer_id = o.customer_id
    )

    SELECT
        customer_unique_id,

        COUNT(DISTINCT order_id) AS total_orders,

        COUNT(total_item_value) AS orders_with_item_data,

        SUM(
            CASE
                WHEN total_item_value IS NULL THEN 1
                ELSE 0
            END
        ) AS orders_missing_item_data,

        ROUND(SUM(total_price), 2) AS total_spending,

        ROUND(AVG(total_item_value), 2) AS avg_order_value,

        SUM(total_items) AS total_items_purchased

    FROM customer_order_details
    GROUP BY customer_unique_id
""")

In [0]:
existing_customer = spark.table(
    'workspace.olist.gold_customer_analytics'
)

print("SQL rows:", gold_customer_sql.count())
print("Existing Gold rows:", existing_customer.count())

print(
    "Distinct customers:",
    gold_customer_sql.select('customer_unique_id').distinct().count()
)

In [0]:
customer_columns = [
    'customer_unique_id',
    'total_orders',
    'orders_with_item_data',
    'orders_missing_item_data',
    'total_spending',
    'avg_order_value',
    'total_items_purchased'
]

sql_difference = gold_customer_sql.select(*customer_columns).exceptAll(
    existing_customer.select(*customer_columns)
)

gold_difference = existing_customer.select(*customer_columns).exceptAll(
    gold_customer_sql.select(*customer_columns)
)

print("SQL records missing from Gold:", sql_difference.count())
print("Gold records missing from SQL:", gold_difference.count())